# Semantic Segmentation for Robot Navigation - Complete Training Pipeline

## 🎯 Project Overview

This notebook trains a **DeepLabV3+** model with **ResNet-50** backbone for semantic segmentation of driving scenes. The model learns to segment images into navigable vs non-navigable areas for autonomous robot navigation.

### Key Features:
- **Model**: DeepLabV3+ with pretrained ResNet-50
- **Dataset**: CamVid (Cambridge driving dataset)
- **Target**: 85%+ mean IoU
- **Applications**: Robot path planning, autonomous navigation

### Notebook Sections:
1. Environment Setup & Installation
2. Dataset Download & Exploration
3. Model Creation & Training
4. Evaluation & Visualization
5. Inference & Cost Map Generation
6. Path Planning with A*

---

## 📦 1. Environment Setup

In [ ]:
# Check if running on Colab
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("✓ Running on Google Colab")
    
    # Clone repository
    !git clone https://github.com/YOUR_USERNAME/robot-navigation-segmentation.git
    %cd robot-navigation-segmentation
else:
    print("✓ Running locally")
    # Ensure we're in the project directory
    import os
    if not os.path.exists('config.py'):
        print("⚠ Please run this notebook from the project root directory")

In [ ]:
# Install dependencies
!pip install -q torch torchvision torchaudio
!pip install -q albumentations opencv-python matplotlib seaborn tqdm tensorboard
!pip install -q networkx scikit-learn pandas wget

print("\n✓ All dependencies installed!")

In [ ]:
# Import libraries
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Check GPU availability
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

if device == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠ No GPU detected. Training will be slow.")

## 📥 2. Dataset Download & Exploration

In [ ]:
# Download CamVid dataset
!python scripts/download_dataset.py

In [ ]:
# Visualize dataset samples
from utils.dataset import CamVidDataset, get_train_transform
from utils.visualization import visualize_prediction
import config

# Load dataset
data_dir = config.RAW_DATA_DIR / "camvid"
dataset = CamVidDataset(
    root_dir=str(data_dir),
    split='train',
    transform=get_train_transform()
)

print(f"Dataset size: {len(dataset)} images")
print(f"Classes: {list(config.CAMVID_CLASSES.values())}")

# Show a few samples
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i in range(6):
    sample = dataset[i * 50]  # Sample every 50th image
    
    # Denormalize image for visualization
    img = sample['image'].permute(1, 2, 0).cpu().numpy()
    img = img * np.array(config.IMAGENET_STD) + np.array(config.IMAGENET_MEAN)
    img = np.clip(img, 0, 1)
    
    axes[i].imshow(img)
    axes[i].set_title(sample['image_name'])
    axes[i].axis('off')

plt.tight_layout()
plt.show()

## 🏗️ 3. Model Creation

In [ ]:
from models.deeplabv3plus import create_model

# Create model
model = create_model(
    num_classes=config.NUM_CLASSES,
    pretrained=True,
    device=device
)

print("\n✓ Model created successfully!")

## 🏋️ 4. Training

Now we'll train the model for 20 epochs. This should take approximately:
- **Colab GPU (T4)**: ~30-40 minutes
- **Colab CPU**: ~4-5 hours (not recommended)

Expected results:
- Training mIoU: 75-85%
- Validation mIoU: 70-80%

In [ ]:
# Option 1: Use the training script (recommended)
!python scripts/train.py

In [ ]:
# Option 2: Train directly in notebook (for more control)
from utils.dataset import create_dataloaders
from scripts.train import Trainer
import torch.nn as nn
import torch.optim as optim

# Create dataloaders
dataloaders = create_dataloaders(
    data_dir=str(data_dir),
    batch_size=config.BATCH_SIZE,
    num_workers=2,
    img_size=config.INPUT_SIZE
)

# Setup training
optimizer = optim.Adam(
    model.get_params(lr=config.LEARNING_RATE, weight_decay=config.WEIGHT_DECAY),
    lr=config.LEARNING_RATE
)

lambda_poly = lambda epoch: (1 - epoch / config.NUM_EPOCHS) ** config.LR_POWER
scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lambda_poly)
criterion = nn.CrossEntropyLoss()

# Create trainer
trainer = Trainer(
    model=model,
    train_loader=dataloaders['train'],
    val_loader=dataloaders['val'],
    optimizer=optimizer,
    scheduler=scheduler,
    criterion=criterion,
    device=device,
    num_classes=config.NUM_CLASSES,
    checkpoint_dir=config.CHECKPOINTS_DIR,
    tensorboard_dir=config.TENSORBOARD_DIR,
    use_amp=config.USE_AMP
)

# Start training
trainer.train(num_epochs=config.NUM_EPOCHS)

## 📊 5. Evaluation & Visualization

In [ ]:
# Load best model
checkpoint_path = config.CHECKPOINTS_DIR / 'best.pth'
checkpoint = torch.load(checkpoint_path)
model.load_state_dict(checkpoint['model_state_dict'])

print(f"✓ Loaded best model")
print(f"  Epoch: {checkpoint['epoch'] + 1}")
print(f"  Best val IoU: {checkpoint['best_val_iou']:.4f}")

In [ ]:
# Evaluate on test set
!python scripts/evaluate.py --checkpoint models/checkpoints/best.pth --split test --max-visualizations 10

In [ ]:
# Visualize predictions
from utils.visualization import visualize_batch

# Get a batch from test set
test_loader = dataloaders['test']
batch = next(iter(test_loader))

images = batch['image'].to(device)
labels = batch['label'].to(device)

# Get predictions
model.eval()
with torch.no_grad():
    outputs = model(images)
    predictions = outputs['out']

# Visualize
visualize_batch(
    images=images,
    predictions=predictions,
    targets=labels,
    max_images=4
)

## 🗺️ 6. Cost Map Generation for Robot Navigation

In [ ]:
def generate_cost_map(segmentation_mask, cost_values):
    """
    Convert segmentation mask to cost map for path planning
    
    Args:
        segmentation_mask: [H, W] with class indices
        cost_values: Dictionary mapping class indices to costs
    
    Returns:
        Cost map [H, W] with navigation costs
    """
    cost_map = np.zeros_like(segmentation_mask, dtype=np.float32)
    
    for class_idx, cost in cost_values.items():
        mask = segmentation_mask == class_idx
        cost_map[mask] = cost
    
    return cost_map


# Generate cost map for a prediction
pred_mask = torch.argmax(predictions[0], dim=0).cpu().numpy()

# Define cost values (lower = more navigable)
cost_values_by_class = {
    3: 1,    # Road - highly navigable
    4: 2,    # Pavement - navigable
    0: 5,    # Sky - neutral
    5: 30,   # Tree - caution
    6: 20,   # Sign - caution
    1: 100,  # Building - obstacle
    2: 100,  # Pole - obstacle
    7: 100,  # Fence - obstacle
    8: 100,  # Car - obstacle
    9: 50,   # Pedestrian - avoid
    10: 50,  # Bicyclist - avoid
    11: 10   # Unlabelled
}

cost_map = generate_cost_map(pred_mask, cost_values_by_class)

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Original image
img_np = images[0].cpu().permute(1, 2, 0).numpy()
img_np = img_np * np.array(config.IMAGENET_STD) + np.array(config.IMAGENET_MEAN)
img_np = np.clip(img_np, 0, 1)
axes[0].imshow(img_np)
axes[0].set_title('Input Image')
axes[0].axis('off')

# Segmentation
from utils.visualization import label_to_color, create_color_map
color_map = create_color_map()
pred_colored = label_to_color(pred_mask, color_map)
axes[1].imshow(pred_colored)
axes[1].set_title('Segmentation')
axes[1].axis('off')

# Cost map
im = axes[2].imshow(cost_map, cmap='hot', vmin=0, vmax=100)
axes[2].set_title('Cost Map (darker = more navigable)')
axes[2].axis('off')
plt.colorbar(im, ax=axes[2], label='Cost')

plt.tight_layout()
plt.show()

print("\n✓ Cost map generated!")
print(f"  Low cost (navigable) area: {(cost_map < 10).sum() / cost_map.size * 100:.1f}%")
print(f"  High cost (obstacle) area: {(cost_map >= 50).sum() / cost_map.size * 100:.1f}%")

## 🛤️ 7. Path Planning with A* Algorithm

In [ ]:
import heapq
from typing import List, Tuple

def astar_path_planning(cost_map, start, goal):
    """
    A* path planning on cost map
    
    Args:
        cost_map: [H, W] cost map
        start: (row, col) start position
        goal: (row, col) goal position
    
    Returns:
        List of (row, col) positions forming the path
    """
    h, w = cost_map.shape
    
    # Heuristic function (Euclidean distance)
    def heuristic(pos1, pos2):
        return np.sqrt((pos1[0] - pos2[0])**2 + (pos1[1] - pos2[1])**2)
    
    # 8-connected neighbors
    neighbors = [(-1,0), (1,0), (0,-1), (0,1), (-1,-1), (-1,1), (1,-1), (1,1)]
    
    # Priority queue: (f_score, g_score, position)
    open_set = [(heuristic(start, goal), 0, start)]
    came_from = {}
    g_score = {start: 0}
    closed_set = set()
    
    while open_set:
        _, current_g, current = heapq.heappop(open_set)
        
        if current == goal:
            # Reconstruct path
            path = [current]
            while current in came_from:
                current = came_from[current]
                path.append(current)
            return path[::-1]
        
        if current in closed_set:
            continue
        
        closed_set.add(current)
        
        # Check neighbors
        for dr, dc in neighbors:
            neighbor = (current[0] + dr, current[1] + dc)
            
            # Check bounds
            if not (0 <= neighbor[0] < h and 0 <= neighbor[1] < w):
                continue
            
            # Skip obstacles (very high cost)
            if cost_map[neighbor] > 80:
                continue
            
            # Calculate cost
            move_cost = 1.414 if abs(dr) + abs(dc) == 2 else 1.0  # Diagonal vs straight
            tentative_g = g_score[current] + move_cost * cost_map[neighbor]
            
            if neighbor not in g_score or tentative_g < g_score[neighbor]:
                came_from[neighbor] = current
                g_score[neighbor] = tentative_g
                f_score = tentative_g + heuristic(neighbor, goal)
                heapq.heappush(open_set, (f_score, tentative_g, neighbor))
    
    return None  # No path found


# Downsample cost map for faster planning
from scipy.ndimage import zoom
small_cost_map = zoom(cost_map, 0.25, order=1)

# Define start and goal positions
h, w = small_cost_map.shape
start = (h - 10, 10)  # Bottom-left
goal = (h - 10, w - 10)  # Bottom-right

print(f"Planning path from {start} to {goal}...")
path = astar_path_planning(small_cost_map, start, goal)

if path:
    print(f"✓ Path found! Length: {len(path)} steps")
    
    # Visualize
    fig, ax = plt.subplots(1, 1, figsize=(12, 8))
    
    # Show cost map
    ax.imshow(small_cost_map, cmap='hot', vmin=0, vmax=100, alpha=0.7)
    
    # Plot path
    if path:
        path_array = np.array(path)
        ax.plot(path_array[:, 1], path_array[:, 0], 'b-', linewidth=3, label='Planned Path')
        ax.plot(start[1], start[0], 'go', markersize=15, label='Start')
        ax.plot(goal[1], goal[0], 'ro', markersize=15, label='Goal')
    
    ax.set_title('A* Path Planning on Cost Map')
    ax.legend()
    ax.axis('off')
    plt.tight_layout()
    plt.show()
    
else:
    print("✗ No path found!")

## 🎉 Summary & Next Steps

### What We Accomplished:
1. ✅ Downloaded and explored CamVid dataset
2. ✅ Trained DeepLabV3+ model for semantic segmentation
3. ✅ Achieved strong mIoU performance
4. ✅ Generated cost maps from segmentation
5. ✅ Implemented A* path planning

### Project Deliverables:
- Trained model checkpoint: `models/checkpoints/best.pth`
- Training curves: `results/training_curves.png`
- Prediction visualizations: `results/predictions/`
- TensorBoard logs: `results/tensorboard/`

### For Your Portfolio:
1. **GitHub**: Upload this project to show employers
2. **README**: Include training results and sample predictions
3. **Demo Video**: Record inference on test images
4. **Report**: Write technical summary of approach and results

### Future Improvements:
- Try different backbones (ResNet-101, EfficientNet)
- Experiment with other datasets (Cityscapes, KITTI)
- Add temporal consistency for video segmentation
- Deploy as REST API for real-time inference
- Integrate with ROS for actual robot control

---

**Good luck with your Hamburg University application! 🎓**